## 1. Load and profile the customer data
Read the raw fields as text to preserve codes, including postal codes with leading zeros. Trim surrounding whitespace and represent blank fields as missing values. The table shows missingness and the number of distinct recorded values in each field.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = project_root / "data"

clientes_raw = pd.read_csv(
    data_dir / "clientes.csv",
    dtype="string",
    keep_default_na=False,
)
clientes = clientes_raw.apply(lambda column: column.str.strip()).replace("", pd.NA)

profile = pd.DataFrame({
    "missing": clientes.isna().sum(),
    "missing (%)": clientes.isna().mean().mul(100).round(1),
    "distinct recorded values": clientes.nunique(),
})

print(f"Customer rows: {len(clientes):,} | Columns: {len(clientes.columns)}")
display(profile)

Customer rows: 1,143 | Columns: 13


,missing,missing (%),distinct recorded values
customer_id,0,0.0,1143
country_or_region,0,0.0,11
commercial_region,114,10.0,37
island,578,50.6,10
municipality,159,13.9,183
city,546,47.8,417
postal_area,71,6.2,271
postal_code,170,14.9,828
street,598,52.3,511
address,6,0.5,1120


## Key insight - Section 1
The source file has 1,143 rows and 13 columns. customer_type is empty in every row. island is missing for 578 customers. Its missingness can be tested as a feature in notebook 03.

company_name has 1,141 distinct values and address has 1,120. The cleaned table retains these fields for inspection. The existing feature-engineering lookup reads only the key, country and island columns.

## 2. Validate the customer ID

`customer_id` connects each purchase to its customer profile. For example, a customer with five purchases has five rows in the transaction data and one row in `clientes.csv`.

Before joining the tables, we check the following.

1. **Every ID is present and contains only digits.** `str.fullmatch()` checks the entire value. `na=False` makes missing IDs fail this check.

2. **IDs are stored as integers.** `pd.to_numeric(...).astype("int64")` converts text such as `"123456"` into the integer `123456`.

3. **Each ID appears only once in the customer table.** `duplicated(keep=False)` flags every row belonging to a repeated ID. Repeated customer profiles could multiply purchase rows during a join.

If a format or uniqueness check fails, the code raises a `ValueError` so we can inspect the affected records before continuing.

When both checks pass, the code prints the number of unique customer IDs and checks for exact duplicate profiles, where every column has the same value.

In [ ]:
raw_ids = clientes["customer_id"]
valid_ids = raw_ids.str.fullmatch(r"[0-9]+", na=False)
if not valid_ids.all():
    raise ValueError(f"Invalid or missing customer IDs: {(~valid_ids).sum()}")

customer_ids = pd.to_numeric(raw_ids).astype("int64")
duplicate_ids = customer_ids.duplicated(keep=False)
if duplicate_ids.any():
    raise ValueError(f"Rows with a repeated customer ID: {duplicate_ids.sum()}")

print(f"Valid and unique customer IDs: {customer_ids.nunique():,}")
print(f"Exact duplicate profiles: {clientes.duplicated().sum()}")

Valid and unique customer IDs: 1,143
Exact duplicate profiles: 0


## 3. Clean the customer fields

We create `clientes_clean` from the customer table and apply the following steps.

1. **Use the validated customer IDs.** Replace `customer_id` with the integer IDs checked in Section 2.

2. **Shorten geographic labels to their codes.** For example, `Country CT10: Land of The Final Version` becomes `CT10`. We do the same for commercial regions (`RG17`) and islands (`IS7`). The codes remain text categories.

3. **Check that shortening preserves the categories.** Every populated label must produce a code, and the number of distinct categories must match before and after extraction. If either check fails, the code raises a `ValueError` for inspection. Missing labels stay missing.

4. **Validate and convert the start year.** Recorded `customer_since_year` values must contain four digits and start with a nonzero digit. We then convert the column to pandas `Int64`, an integer type that also supports missing values. Section 4 checks whether these years agree with purchase dates.

5. **Remove the empty customer-type column.** We drop `customer_type` when every value is missing.

The final table compares the number of distinct geographic categories before and after shortening. Matching counts confirm that the shorter labels preserve the category distinctions.

In [4]:
clientes_clean = clientes.copy()
clientes_clean["customer_id"] = customer_ids

checks = {}
label_changes = []

code_patterns = {
    "country_or_region": r"^Country (CT[0-9]+): .+$",
    "commercial_region": r"^Region (RG[0-9]+): .+$",
    "island": r"^Island (IS[0-9]+): .+$",
}

for column, pattern in code_patterns.items():
    labels = clientes[column]
    codes = labels.str.extract(pattern, expand=False)
    invalid = labels.notna() & codes.isna()

    if invalid.any() or labels.nunique() != codes.nunique():
        raise ValueError(
            f"{column}: {invalid.sum()} invalid labels; "
            f"{labels.nunique()} original categories, "
            f"{codes.nunique()} codes"
        )

    clientes_clean[column] = codes

    checks[f"{column}: labels without a valid code"] = {
        "before": int(invalid.sum()),
        "after": int((labels.notna() & codes.isna()).sum()),
    }

    label_changes.append({
        "column": column,
        "labels shortened": int(labels.ne(codes).fillna(False).sum()),
        "categories before": labels.nunique(),
        "categories after": codes.nunique(),
        "missing before": labels.isna().sum(),
        "missing after": codes.isna().sum(),
    })

# Check the year format and convert to a nullable integer.
year = clientes["customer_since_year"]
invalid_year = (
    year.notna()
    & ~year.str.fullmatch(r"[1-9][0-9]{3}", na=False)
)

if invalid_year.any():
    raise ValueError(
        f"Invalid customer start years: {invalid_year.sum()}"
    )

clientes_clean["customer_since_year"] = pd.to_numeric(year).astype("Int64")
year_after = clientes_clean["customer_since_year"].astype("string")

checks["customer_since_year: invalid format"] = {
    "before": int(invalid_year.sum()),
    "after": int((
        year_after.notna()
        & ~year_after.str.fullmatch(r"[1-9][0-9]{3}", na=False)
    ).sum()),
}

# Remove customer_type when the entire column is missing.
empty_before = int(clientes_clean.isna().all().sum())

if clientes_clean["customer_type"].isna().all():
    clientes_clean = clientes_clean.drop(columns="customer_type")

checks["Entirely empty columns"] = {
    "before": empty_before,
    "after": int(clientes_clean.isna().all().sum()),
}

print("Data-quality checks")
display(pd.DataFrame.from_dict(checks, orient="index"))

print("Label standardisation and missingness")
display(pd.DataFrame(label_changes).set_index("column"))

Data-quality checks


,before,after
country_or_region: labels without a valid code,0,0
commercial_region: labels without a valid code,0,0
island: labels without a valid code,0,0
customer_since_year: invalid format,0,0
Entirely empty columns,1,0


Label standardisation and missingness


,labels shortened,categories before,categories after,missing before,missing after
column,,,,,
country_or_region,1143,11,11,0,0
commercial_region,1029,37,37,114,114
island,565,10,10,578,578



**Found**

- All recorded geographic labels and customer start years passed the format checks.
- `customer_type` was entirely empty and was removed.
- We shortened 1,143 country labels, 1,029 region labels and 565 island labels, preserving all **11 country, 37 region and 10 island categories**.
- There are **114 missing commercial regions** and **578 missing island values**. Their patterns are examined in Section 3.1.

**For preprocessing**

- Use the shortened codes as categorical features.
- Evaluate missing-value indicators and any imputation within the modelling pipeline.
- The cleaned customer table contains **1,143 rows and 12 columns**.

### 3.1 Missingness patterns

We examine whether missing address information is associated with geography and whether several fields are missing together in the same customer profiles.

The code produces three tables.

1. **Missing values by country group.** We compare `CT10`, the largest country group, with the remaining countries combined. For each address field, the table shows the percentage of customers with a missing value. The `customers` column gives each group's size.

2. **Island coverage by commercial region.** Regions with at least one recorded island are shown individually. The remaining regions are grouped as `Other recorded regions`, and customers without a region appear under `Region missing`. The table counts recorded and missing island values.

3. **Combinations of missing fields.** We count customers sharing the same missingness pattern across `commercial_region`, `municipality` and `island`. `True` means the field is missing, and `False` means it is filled.

These patterns help identify potentially structural missingness and gaps that need further investigation. Interpreting their causes requires understanding how the fields were collected.

In [5]:
address_fields = [
    "commercial_region", "island", "municipality", "city",
    "postal_area", "postal_code", "street", "address",
]
country = clientes_clean["country_or_region"]
country_group = country.where(country.eq("CT10"), "Other countries")
country_group = country_group.rename("Country group")

missing_by_country = (
    clientes_clean[address_fields]
    .isna()
    .groupby(country_group)
    .mean()
    .mul(100)
    .round(1)
)
missing_by_country.insert(0, "customers", country_group.value_counts())
display(missing_by_country)

region = clientes_clean["commercial_region"]
regions_with_islands = region[clientes_clean["island"].notna()].dropna().unique()
region_group = region.where(region.isin(regions_with_islands), "Other recorded regions")
region_group = region_group.mask(region.isna(), "Region missing")
island_by_region = (
    clientes_clean["island"]
    .groupby(region_group.rename("Commercial region"))
    .agg(customers="size", islands_recorded="count")
)
island_by_region["islands_missing"] = (
    island_by_region["customers"] - island_by_region["islands_recorded"]
)
display(island_by_region)

# True marks a missing field in each combination.
location_missing = clientes_clean[["commercial_region", "municipality", "island"]].isna()
display(location_missing.value_counts().rename("customers").to_frame())

,customers,commercial_region,island,municipality,city,postal_area,postal_code,street,address
Country group,,,,,,,,,
CT10,1079,8.8,47.6,8.8,48.0,0.6,15.4,50.0,0.6
Other countries,64,29.7,100.0,100.0,43.8,100.0,6.2,90.6,0.0


,customers,islands_recorded,islands_missing
Commercial region,,,
Other recorded regions,464,0,464
RG17,519,519,0
RG29,46,46,0
Region missing,114,0,114


customers
commercial_region municipality island           
False             False        False         565
                               True          419
True              True         True          114
False             True         True           45

### Key insight - Section 3.1

**Found**

- All **64 customers outside CT10** have missing `island`, `municipality` and `postal_area`. Among CT10 customers, the corresponding missing rates are **47.6%, 8.8% and 0.6%**.
- `island` is recorded for every customer in **RG17 (519)** and **RG29 (46)**. The remaining **578 customers** have no recorded island.
- All **114 customers with a missing commercial region** also lack municipality and island information. Another **45 customers** have a recorded region but lack both municipality and island.
- Missingness is associated with observed geography and occurs jointly across fields. This suggests that some blanks may be **structural**, meaning the field does not apply to that location. The source definitions are needed to confirm this interpretation.

**For preprocessing**

- Preserve the missing values in the cleaned customer lookup.
- Test `island_missing` together with country and commercial region to assess whether it adds predictive information.
- Fit any categorical imputation inside each training fold.

### 3.2 Check geographic relationships

We compare related geographic fields to find combinations that need closer inspection. For example, we check whether the same postal code is linked to several municipalities or the same address is linked to different cities.

For each pair, we group profiles by country and the first field, then count the distinct values in the linked field. A group is flagged when it contains more than one recorded value. Only profiles with both fields available contribute to this comparison.

The code creates three tables.

- **`geography_checks`** summarises how many profiles and groups were compared, and how many groups were flagged.
- **`geography_flags`** lists the flagged labels and their number of distinct linked values.
- **`geography_review`** stores the customer profiles belonging to those groups for investigation in the next section.

We inspect the surrounding fields before deciding on a correction. Some relationships can legitimately involve several values, such as a postal area covering multiple municipalities.

In [6]:
geographic_pairs = [
    ("municipality", "commercial_region"),
    ("city", "municipality"),
    ("postal_code", "postal_area"),
    ("postal_code", "municipality"),
    ("postal_area", "municipality"),
    ("address", "city"),
    ("address", "postal_code"),
]
geographic_checks = []
geographic_flags = []
geographic_review = []
profile_fields = [
    "customer_id", "country_or_region", "commercial_region", "island",
    "municipality", "city", "postal_area", "postal_code", "street", "address",
]
for field, linked_field in geographic_pairs:
    keys = ["country_or_region", field]
    known = clientes_clean[[*keys, linked_field]].dropna()
    distinct_links = known.groupby(keys)[linked_field].nunique()
    multiple_links = distinct_links[distinct_links.gt(1)]

    geographic_checks.append({
        "field": field,
        "linked field": linked_field,
        "comparable profiles": len(known),
        "groups checked": len(distinct_links),
        "groups with multiple values": len(multiple_links),
    })

    flagged = multiple_links.rename("distinct values").reset_index()
    flagged[field] = flagged[field].str.split(":", n=1).str[0]
    flagged = flagged.rename(columns={field: "label"})
    flagged.insert(0, "field", field)
    flagged.insert(1, "linked field", linked_field)
    geographic_flags.append(flagged)

    profile_keys = pd.MultiIndex.from_frame(clientes_clean[keys])
    in_flagged_group = profile_keys.isin(multiple_links.index)
    profiles = clientes_clean.loc[in_flagged_group, profile_fields]
    geographic_review.append(profiles.assign(field=field, linked_field=linked_field))

geography_checks = pd.DataFrame(geographic_checks)
geography_flags = pd.concat(geographic_flags, ignore_index=True)
geography_review = pd.concat(geographic_review, ignore_index=True)
display(geography_checks)
display(geography_flags)

,field,linked field,comparable profiles,groups checked,groups with multiple values
0,municipality,commercial_region,984,183,0
1,city,municipality,546,367,0
2,postal_code,postal_area,913,771,0
3,postal_code,municipality,851,714,1
4,postal_area,municipality,982,252,6
5,address,city,597,589,4
6,address,postal_code,970,956,3


,field,linked field,country_or_region,label,distinct values
0,postal_code,municipality,CT10,Postal code PC147,2
1,postal_area,municipality,CT10,Postal area PA106,2
2,postal_area,municipality,CT10,Postal area PA119,2
3,postal_area,municipality,CT10,Postal area PA30,3
4,postal_area,municipality,CT10,Postal area PA39,2
5,postal_area,municipality,CT10,Postal area PA61,2
6,postal_area,municipality,CT10,Postal area PA84,2
7,address,city,CT10,Address AD125,2
8,address,city,CT10,Address AD251,2
9,address,city,CT10,Address AD270,2


What to review
- The comparable records give one postal area per postal code. PC147 is linked to two municipalities, MU117 and MU119, within the same country code. Six postal areas are linked to multiple municipalities.
- Four address labels appear with multiple city values and three with multiple postal codes. A review of the source definitions or records would be needed to decide which values to correct.

### 3.3 Inspect the profiles behind the flags

We examine the customer records to understand what caused each flag and whether the surrounding geographic information agrees.

1. **Count the affected customers.** We count distinct customers for each flagged relationship and across all flagged groups. A customer can appear in several checks, so the overall count avoids counting that customer more than once.

2. **Examine the flagged postal areas.** For each area, we show the number of customers, how many have a recorded municipality, and the number of distinct municipalities and commercial regions. This helps assess whether an area covers several municipalities within the same region.

3. **Compare individual customer profiles.** We display the records associated with flagged postal codes and repeated addresses. Comparing their regions, municipalities, cities and postal codes shows exactly which fields agree and which differ. Geographic labels are shortened to their codes for readability.

4. **Check the postal-code-to-city relationship.** Across all profiles with the required fields available, we count how many postal codes are associated with multiple cities within the same country. This helps assess whether a postal-code lookup could reliably fill missing city values.

These comparisons provide evidence for documenting each case and deciding whether a correction requires further source information.

In [7]:
flagged_customer_counts = (
    geography_review.groupby(["field", "linked_field"])["customer_id"]
    .nunique()
    .rename("customer profiles")
    .to_frame()
)
display(flagged_customer_counts)
print(f"Distinct customers across flagged groups: {geography_review['customer_id'].nunique():,}")

area_profiles = geography_review.loc[geography_review["field"].eq("postal_area")]
postal_area_review = (
    area_profiles.groupby("postal_area")
    .agg(
        customers=("customer_id", "nunique"),
        municipality_recorded=("municipality", "count"),
        distinct_municipalities=("municipality", "nunique"),
        distinct_recorded_regions=("commercial_region", "nunique"),
    )
    .sort_values("customers", ascending=False)
)
postal_area_review.index = postal_area_review.index.str.split(":", n=1).str[0]
display(postal_area_review)

local_cases = geography_review.loc[
    geography_review["field"].isin(["postal_code", "address"]), profile_fields
].drop_duplicates()
for column in profile_fields[2:]:
    local_cases[column] = local_cases[column].str.split(":", n=1).str[0].str.rsplit(n=1).str[-1]
display(local_cases.sort_values(["address", "city", "postal_code"]))

postal_city = clientes_clean.dropna(subset=["country_or_region", "postal_code", "city"])
cities_per_code = postal_city.groupby(["country_or_region", "postal_code"])["city"].nunique()
print(f"Postal codes with multiple recorded cities: {cities_per_code.gt(1).sum()} "
      f"of {len(cities_per_code)} comparable codes")

customer profiles
field       linked_field                   
address     city                          9
            postal_code                   6
postal_area municipality                 40
postal_code municipality                  2

Distinct customers across flagged groups: 53


,customers,municipality_recorded,distinct_municipalities,distinct_recorded_regions
postal_area,,,,
Postal area PA84,19,14,2,1
Postal area PA119,6,6,2,1
Postal area PA39,6,4,2,1
Postal area PA30,4,4,3,1
Postal area PA61,3,2,2,1
Postal area PA106,2,2,2,1


,customer_id,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address
47,149036,CT10,RG17,IS6,MU2,CY56,PA262,<NA>,<NA>,AD125
46,684173,CT10,RG17,IS6,MU2,CY64,PA262,PC696,<NA>,AD125
45,386004,CT10,RG17,IS4,MU6,CY89,PA266,<NA>,ST69,AD251
44,554016,CT10,RG17,IS4,MU6,CY96,PA266,PC748,ST69,AD251
42,806571,CT10,RG17,IS4,MU7,CY108,PA268,PC799,<NA>,AD270
43,550457,CT10,RG17,IS4,MU7,CY113,PA268,PC803,<NA>,AD270
54,425287,CT10,RG17,IS4,MU15,CY176,PA267,PC772,<NA>,AD461
53,838689,CT10,RG17,IS4,MU15,<NA>,PA267,PC781,<NA>,AD461
49,746863,CT10,RG17,IS6,MU16,CY185,PA261,PC674,<NA>,AD489
48,238158,CT10,RG17,IS6,MU16,CY187,PA261,PC674,<NA>,AD489


Postal codes with multiple recorded cities: 38 of 451 comparable codes


### Key insight - Section 3.3

**Found**

- **53 distinct customers belong to flagged geographic groups.** These consist of **40 customers in six postal-area groups** and **13 customers sharing six address labels**.

- **The six postal areas are linked to multiple municipalities.** Of their 40 customer profiles, 32 have a recorded municipality and eight have it missing. Each postal area has only one recorded commercial region. This is consistent with postal areas covering several municipalities.

- **Postal code `PC147` is linked to two municipalities**, `MU117` and `MU119`. The two customers share country `CT10`, region `RG28` and postal area `PA61`, with different streets and addresses. They are already included in the 40 postal-area profiles.

- **Six repeated address labels have differing city or postal-code values.** Within each address label, the recorded country, region, island, municipality and postal area agree. The differences concern more detailed location information.

| Address label | Customers | Differences found |
|---|---:|---|
| `AD125` | 2 | Cities `CY56` and `CY64`. |
| `AD251` | 2 | Cities `CY89` and `CY96`, despite sharing street `ST69`. |
| `AD270` | 2 | City/postal-code combinations `CY108 / PC799` and `CY113 / PC803`. |
| `AD461` | 2 | Postal codes `PC772` and `PC781`. |
| `AD489` | 3 | Cities `CY185` and `CY187`, with the same postal code `PC674`. |
| `AD492` | 2 | Postal codes `PC659` and `PC674`. |

The address checks flag nine customers through city differences and six through postal-code differences. The two customers at `AD270` appear in both checks, giving **13 unique customers**.

Across the wider customer table, **38 of 451 comparable postal codes (8.4%)** have multiple recorded cities within the same country. Reconstructing a missing city from its postal code would therefore be ambiguous for these codes.

**For preprocessing**

- Document the flagged relationships and retain the recorded values until a correction is supported by source evidence.
- Check that a geographic lookup gives a unique, consistent result before using it to fill a missing field.
- Proceed with the cleaned export and evaluate the geographic features during validation.

## 4. Validate the customer lookup against the purchase files

We check the cleaned customer lookup against `train.csv` and `test.csv` using `customer_id`.

- **Match coverage.** Count purchase rows with and without a matching customer profile. The left join includes every purchase in this check.
- **Start-year consistency.** For matched profiles with both years available, check whether the purchase occurred before `customer_since_year`.
- **Unavailable year checks.** Count rows where the comparison cannot be performed because the customer profile, purchase date or start year is missing.

The 65 undated training records still contribute to match coverage. Their missing purchase dates make the start-year comparison unavailable.

In [9]:
customer_year = clientes_clean[["customer_id", "customer_since_year"]]
checks = []

for split in ("train", "test"):
    purchases = pd.read_csv(
        Path(data_dir) / f"{split}.csv",
        usecols=["customer_id", "purchase_date"],
        parse_dates=["purchase_date"],
        dtype={"customer_id": "Int64"},
    )
    joined = purchases.merge(
        customer_year,
        on="customer_id",
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    matched = joined["_merge"].eq("both")
    purchase_year = joined["purchase_date"].dt.year
    since_year = joined["customer_since_year"]

    comparable = (
        matched
        & purchase_year.notna()
        & since_year.notna()
    )
    before_start = purchase_year.loc[comparable].lt(
        since_year.loc[comparable]
    )

    checks.append({
        "dataset": split,
        "purchase rows": len(purchases),
        "matched rows": int(matched.sum()),
        "unmatched rows": int((~matched).sum()),
        "start-year checks": int(comparable.sum()),
        "start-year checks unavailable": int((~comparable).sum()),
        "purchases before start year": int(before_start.sum()),
    })

join_checks = pd.DataFrame(checks).set_index("dataset")
display(join_checks)

,purchase rows,matched rows,unmatched rows,start-year checks,start-year checks unavailable,purchases before start year
dataset,,,,,,
train,6534,6534,0,6469,65,0
test,986,986,0,986,0,0


## Key insight - Section 4
All 6,534 train rows and 986 test rows match a customer profile. The join preserves the number of purchase rows. All comparable purchase years are at or after the corresponding customer start year.

In [10]:
from pathlib import Path

processed_dir = Path(data_dir) / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "clientes_clean.csv"
clientes_clean.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())
print("Customers:", len(clientes_clean), "| Columns:", len(clientes_clean.columns))

Saved: C:\Users\Duarte\Desktop\git\ETAI-Group5\data\processed\clientes_clean.csv
Customers: 1143 | Columns: 12
